# VRDU Ad-buy Form CBA Experiment

**Domain**: Political Advertising Forms (DeepForm/VRDU)  
**Method**: Vision extraction — LLM reads ad-buy invoice page images and maps values to canonical concepts  
**Models**: Claude Haiku 4.5, GPT-4o-mini  
**Scoring**: Value-first CBA matching

Political advertising disclosure forms (FCC filings) contain multiple same-type fields:
4+ date fields (flight_from, flight_to, plus line-item program dates), multiple dollar amounts
(gross_amount plus per-line-item sub_amounts), and several organization names (advertiser, agency, product).
The document-level fields are surrounded by dozens of line-item values that serve as natural distractors.

Key confusable pairs: flight_from ↔ flight_to (adjacent dates), advertiser ↔ product (often near-identical text),
advertiser ↔ agency (both org names), gross_amount vs line-item sub_amounts (total vs individual).

Source: [VRDU (Google Research)](https://github.com/google-research-datasets/vrdu) — 641 real FCC ad-buy invoices.

In [ ]:
import subprocess, sys, os

for pkg in ["anthropic", "openai", "pymupdf", "Pillow"]:
    mod = {"Pillow": "PIL", "pymupdf": "fitz"}.get(pkg, pkg)
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--break-system-packages", "-q"])

import anthropic, openai, json, time, random, io, base64, gzip, shutil
import fitz  # PyMuPDF
from PIL import Image as PILImage
from collections import defaultdict, Counter
from google.colab import userdata


ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# --- Download VRDU ad-buy forms ---
VRDU_BASE = "/tmp/vrdu"
VRDU_PATH = os.path.join(VRDU_BASE, "ad-buy-form", "main")
PDF_DIR = os.path.join(VRDU_PATH, "pdfs")

if not os.path.isdir(PDF_DIR) or len(os.listdir(PDF_DIR)) < 600:
    print("Downloading VRDU dataset (ad-buy forms)...")
    print("This is ~200 MB and may take a few minutes on first run.")
    if os.path.exists(VRDU_BASE):
        shutil.rmtree(VRDU_BASE)
    # Shallow clone — downloads only latest commit, all files
    subprocess.run([
        "git", "clone", "--depth=1",
        "https://github.com/google-research-datasets/vrdu.git", VRDU_BASE
    ], check=True)
    print("Clone complete.")
else:
    print("VRDU ad-buy data already present.")

# Verify
n_pdfs = len([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")])
assert n_pdfs >= 600, f"Expected ~641 PDFs, found {n_pdfs}. Clone may have failed."

# Decompress dataset.jsonl.gz if needed
dataset_path = os.path.join(VRDU_PATH, "dataset.jsonl")
if not os.path.exists(dataset_path):
    gz_path = dataset_path + ".gz"
    with gzip.open(gz_path, 'rb') as f_in:
        with open(dataset_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"Decompressed dataset.jsonl")

print(f"VRDU path: {VRDU_PATH}")
print(f"PDFs: {n_pdfs} files")
print("Setup complete")

VRDU ad-buy data already present.
VRDU path: /tmp/vrdu/ad-buy-form/main
PDFs: 641 files
Setup complete


In [ ]:
# Load VRDU ad-buy dataset from jsonl
dataset_path = os.path.join(VRDU_PATH, "dataset.jsonl")
if not os.path.exists(dataset_path):
    gz_path = dataset_path + ".gz"
    with gzip.open(gz_path, 'rb') as f_in:
        with open(dataset_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"Decompressed dataset.jsonl")

# 9 unrepeated (document-level) concepts to evaluate
EVAL_CONCEPTS = [
    "advertiser",     # Political committee/campaign buying ads
    "agency",         # Media buying agency (intermediary)
    "contract_num",   # Contract/order/invoice number
    "flight_from",    # Overall campaign start date
    "flight_to",      # Overall campaign end date
    "gross_amount",   # Total gross amount for entire order
    "product",        # Product/campaign label
    "tv_address",     # TV station mailing address
    "property",       # TV station call sign (e.g., KMSP)
]

# --- Diagnostic: check PDF directory ---
print(f"PDF_DIR: {PDF_DIR}")
print(f"PDF_DIR exists: {os.path.isdir(PDF_DIR)}")
if os.path.isdir(PDF_DIR):
    pdf_files_on_disk = [f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")]
    print(f"PDFs on disk: {len(pdf_files_on_disk)}")
else:
    pdf_files_on_disk = []
    print("WARNING: PDF directory not found!")

# Parse dataset
all_docs = []
total_parsed = 0
enough_concepts = 0
pdf_found = 0

with open(dataset_path) as f:
    for line in f:
        doc = json.loads(line)
        total_parsed += 1
        gt = {}
        for ann in doc["annotations"]:
            entity_name = ann[0]
            if isinstance(entity_name, str) and entity_name in EVAL_CONCEPTS:
                values = ann[1]
                if values:
                    text = values[0][0].strip()
                    if text and entity_name not in gt:
                        gt[entity_name] = text

        n_pages = len(doc["ocr"]["pages"])
        pdf_path = os.path.join(PDF_DIR, doc["filename"])

        has_concepts = len(gt) >= 5
        has_pdf = os.path.exists(pdf_path)

        if has_concepts:
            enough_concepts += 1
        if has_pdf:
            pdf_found += 1

        if has_concepts and has_pdf:
            all_docs.append({
                "filename": doc["filename"],
                "pdf_path": pdf_path,
                "ground_truth": gt,
                "n_pages": n_pages,
            })

print(f"\n--- Loading Summary ---")
print(f"Total docs in jsonl: {total_parsed}")
print(f"Docs with >= 5 concepts: {enough_concepts}")
print(f"Docs with PDF on disk: {pdf_found}")
print(f"Docs passing both filters: {len(all_docs)}")

if len(all_docs) == 0 and enough_concepts > 0 and pdf_found == 0:
    print(f"\nDIAGNOSTIC: Concepts OK but no PDFs found.")
    print(f"First jsonl filename: {json.loads(open(dataset_path).readline())['filename']}")
    if pdf_files_on_disk:
        print(f"First PDF on disk: {pdf_files_on_disk[0]}")
    else:
        print("No PDFs in PDF_DIR — git clone may not have downloaded them.")
        print("Try running: !git clone --depth=1 https://github.com/google-research-datasets/vrdu.git /tmp/vrdu")
elif len(all_docs) > 0:
    print(f"\nConcept coverage:")
    for c in EVAL_CONCEPTS:
        n = sum(1 for d in all_docs if c in d["ground_truth"])
        print(f"  {c:<20s}: {n}/{len(all_docs)} ({n/len(all_docs):.0%})")
    print(f"\nSample GT (first doc):")
    for k, v in all_docs[0]["ground_truth"].items():
        print(f"  {k}: {v}")

PDF_DIR: /tmp/vrdu/ad-buy-form/main/pdfs
PDF_DIR exists: True
PDFs on disk: 641

--- Loading Summary ---
Total docs in jsonl: 641
Docs with >= 5 concepts: 629
Docs with PDF on disk: 641
Docs passing both filters: 629

Concept coverage:
  advertiser          : 626/629 (100%)
  agency              : 276/629 (44%)
  contract_num        : 620/629 (99%)
  flight_from         : 537/629 (85%)
  flight_to           : 535/629 (85%)
  gross_amount        : 620/629 (99%)
  product             : 607/629 (97%)
  tv_address          : 534/629 (85%)
  property            : 591/629 (94%)

Sample GT (first doc):
  property: KMSP
  tv_address: 4614 Collection Center Drive
Chicago, IL 60693
  advertiser: Michael Bloomberg 2020, Inc
  product: MIKE BLOOMBERG 2020 INC
  contract_num: 950658
  flight_to: 03/29/20
  flight_from: 12/30/19
  gross_amount: $5,625.00


In [ ]:
ADBUY_ONTOLOGY = {
    "families": {
        "organizations": ["advertiser", "agency", "product"],
        "media": ["property", "tv_address"],
        "dates": ["flight_from", "flight_to"],
        "financial": ["gross_amount", "contract_num"],
    }
}

CONCEPT_TO_FAMILY = {c: f for f, cs in ADBUY_ONTOLOGY["families"].items() for c in cs}

ADBUY_SYSTEM_PROMPT = """You are a document AI extraction system. You extract structured data from political advertising disclosure form images (FCC filings).

Given an image (or multiple pages) of an ad-buy form/invoice, extract values for the following canonical concept keys.

CANONICAL CONCEPT KEYS:
- advertiser: Name of the political committee, campaign, or organization buying the ads
- agency: Name of the media buying agency (intermediary firm)
- contract_num: Contract number, order number, or invoice number
- flight_from: Start date of the overall advertising campaign/flight period
- flight_to: End date of the overall advertising campaign/flight period
- gross_amount: Total gross amount for the entire order/invoice
- product: Product or campaign name (may be similar to advertiser name)
- tv_address: Mailing/remit address of the TV station
- property: TV station call sign (e.g., KMSP, WJLA, WTTG)

IMPORTANT DISAMBIGUATION RULES:
- flight_from/flight_to are the OVERALL campaign dates (often labeled "Order Flight" or "Flight Dates"), NOT individual program/spot dates
- gross_amount is the TOTAL amount for the entire invoice, NOT individual line item amounts
- advertiser is the buyer/committee name, agency is the intermediary firm that placed the buy, product is the campaign label
- contract_num is the primary order/contract/invoice number
- property is the station call sign (e.g., "KMSP"), NOT the station name or channel number

RULES:
1. Return ONLY a JSON object with exactly these 9 keys.
2. If a field is not present or you cannot determine it, use "N/A".
3. Dates: use the format shown on the form (e.g., "12/30/19" or "03/29/20").
4. Dollar amounts: include $ and commas as shown (e.g., "$5,625.00").
5. Return ONLY valid JSON, no other text."""

print(f"Ontology: {len(EVAL_CONCEPTS)} concepts, {len(ADBUY_ONTOLOGY['families'])} families")
for fam, concepts in ADBUY_ONTOLOGY["families"].items():
    print(f"  {fam}: {concepts}")

Ontology: 9 concepts, 4 families
  organizations: ['advertiser', 'agency', 'product']
  media: ['property', 'tv_address']
  dates: ['flight_from', 'flight_to']
  financial: ['gross_amount', 'contract_num']


In [ ]:
SAMPLE_SIZE = 50
random.seed(42)

indices = list(range(len(all_docs)))
random.shuffle(indices)
selected = sorted(indices[:SAMPLE_SIZE])

samples = []
for i, idx in enumerate(selected):
    doc = all_docs[idx]
    samples.append({
        "doc_id": f"adbuy_{i:04d}",
        "filename": doc["filename"],
        "pdf_path": doc["pdf_path"],
        "ground_truth": doc["ground_truth"],
        "n_pages": doc["n_pages"],
    })

# Stats
filled = Counter()
for s in samples:
    for c in EVAL_CONCEPTS:
        if c in s["ground_truth"]:
            filled[c] += 1

print(f"Sampled {len(samples)} ad-buy forms")
print(f"\nField fill rates:")
for c in EVAL_CONCEPTS:
    print(f"  {c:<20s}: {filled[c]}/{SAMPLE_SIZE} ({filled[c]/SAMPLE_SIZE:.0%})")

page_dist = Counter(s["n_pages"] for s in samples)
print(f"\nPage count distribution:")
for k in sorted(page_dist.keys()):
    print(f"  {k} pages: {page_dist[k]} docs")

Sampled 50 ad-buy forms

Field fill rates:
  advertiser          : 50/50 (100%)
  agency              : 17/50 (34%)
  contract_num        : 50/50 (100%)
  flight_from         : 43/50 (86%)
  flight_to           : 43/50 (86%)
  gross_amount        : 50/50 (100%)
  product             : 49/50 (98%)
  tv_address          : 45/50 (90%)
  property            : 48/50 (96%)

Page count distribution:
  1 pages: 14 docs
  2 pages: 9 docs
  3 pages: 17 docs
  4 pages: 7 docs
  6 pages: 1 docs
  8 pages: 1 docs
  11 pages: 1 docs


In [ ]:
def normalize_value(v):
    """Normalize a value for comparison."""
    v = str(v).strip().lower()
    v = v.replace("$", "").replace(",", "").strip()
    v = " ".join(v.split())  # collapse whitespace
    # Normalize date separators
    v = v.replace("-", "/")
    # Try numeric normalization
    try:
        numeric = float(v)
        v = f"{numeric:.2f}"
    except ValueError:
        pass
    return v

def score_document(gt, pred):
    """Score a single document with value-first CBA matching."""
    pred_norm = {k: normalize_value(v) for k, v in pred.items()}
    pred_values = {v for v in pred_norm.values() if v != normalize_value("N/A")}

    per_field = {}
    field_correct = 0
    cba_correct = 0
    scored = []

    for concept in EVAL_CONCEPTS:
        gt_val = gt.get(concept)
        if gt_val is None:
            continue  # Skip concepts without GT

        gt_norm = normalize_value(gt_val)
        pred_val = normalize_value(pred.get(concept, "N/A"))

        scored.append(concept)

        # CBA: exact concept match
        cba_match = (gt_norm == pred_val)

        # Field Recall: value found anywhere in predictions
        field_match = (gt_norm in pred_values)

        if cba_match:
            cba_correct += 1
        if field_match:
            field_correct += 1

        # Track misbinding target
        bound_to = None
        if field_match and not cba_match:
            for k, v in pred.items():
                if normalize_value(v) == gt_norm and k != concept:
                    bound_to = k
                    break

        per_field[concept] = {
            "gt_value": gt_norm,
            "pred_value": pred_val,
            "field_match": field_match,
            "cba_match": cba_match,
            "misbinding": field_match and not cba_match,
            "bound_to": bound_to,
        }

    total = len(scored)
    if total == 0:
        return {"field_recall": 0, "cba_strict": 0, "cba_soft": 0, "delta": 0,
                "total": 0, "per_field": {}, "misbinding_count": 0}

    misbinding_count = sum(1 for d in per_field.values() if d["misbinding"])

    # CBA-soft: same family = 0.5 credit
    cba_soft_score = 0
    for concept, detail in per_field.items():
        if detail["cba_match"]:
            cba_soft_score += 1.0
        elif detail["misbinding"] and detail["bound_to"]:
            gt_fam = CONCEPT_TO_FAMILY.get(concept, "")
            pred_fam = CONCEPT_TO_FAMILY.get(detail["bound_to"], "")
            cba_soft_score += 0.5 if gt_fam == pred_fam else 0.0

    return {
        "field_recall": field_correct / total,
        "cba_strict": cba_correct / total,
        "cba_soft": cba_soft_score / total,
        "delta": (field_correct - cba_correct) / total,
        "total": total,
        "scored_concepts": scored,
        "per_field": per_field,
        "misbinding_count": misbinding_count,
    }

print(f"Scoring functions ready ({len(EVAL_CONCEPTS)} concepts)")

Scoring functions ready (9 concepts)


In [ ]:
MAX_PAGES = 3  # Send first 3 pages to vision API
RENDER_DPI = 200

def pdf_to_images(pdf_path, max_pages=MAX_PAGES, dpi=RENDER_DPI):
    """Convert PDF pages to PIL images using PyMuPDF."""
    doc = fitz.open(pdf_path)
    images = []
    for page_num in range(min(len(doc), max_pages)):
        page = doc[page_num]
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        img = PILImage.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)
    doc.close()
    return images

def encode_image(img, max_width=1024):
    """Resize and base64-encode a PIL image."""
    if img.width > max_width:
        ratio = max_width / img.width
        img = img.resize((max_width, int(img.height * ratio)), PILImage.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")

def extract_anthropic(pdf_path, model_id):
    """Extract via Anthropic vision API with multi-page support."""
    images = pdf_to_images(pdf_path)
    content = []
    for img in images:
        b64 = encode_image(img)
        content.append({"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}})
    content.append({"type": "text", "text": "Extract all document-level fields from this ad-buy form."})

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    message = client.messages.create(
        model=model_id,
        max_tokens=1024,
        system=ADBUY_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": content}],
    )
    raw = message.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

def extract_openai(pdf_path, model_id):
    """Extract via OpenAI vision API with multi-page support."""
    images = pdf_to_images(pdf_path)
    content = []
    for img in images:
        b64 = encode_image(img)
        content.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}})
    content.append({"type": "text", "text": "Extract all document-level fields from this ad-buy form."})

    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=model_id,
        max_tokens=1024,
        messages=[
            {"role": "system", "content": ADBUY_SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ],
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

MODELS = {
    "haiku": {"provider": "anthropic", "model_id": "claude-haiku-4-5-20251001"},
    "gpt4o-mini": {"provider": "openai", "model_id": "gpt-4o-mini"},
}

def extract_adbuy(pdf_path, model_name):
    """Route to correct provider."""
    cfg = MODELS[model_name]
    if cfg["provider"] == "anthropic":
        return extract_anthropic(pdf_path, cfg["model_id"])
    else:
        return extract_openai(pdf_path, cfg["model_id"])

print(f"Extraction functions ready. Models: {list(MODELS.keys())}")
print(f"Samples available: {len(samples)}")

# Quick test render
if samples:
    test_imgs = pdf_to_images(samples[0]["pdf_path"])
    print(f"Test render: {len(test_imgs)} pages, first page {test_imgs[0].size}")
else:
    print("ERROR: No samples loaded. Check cell 2 output — likely PDF paths don't exist.")
    print(f"  PDF_DIR exists: {os.path.isdir(PDF_DIR)}")
    if os.path.isdir(PDF_DIR):
        pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]
        print(f"  PDFs in directory: {len(pdf_files)}")
        if pdf_files:
            print(f"  Sample PDF name: {pdf_files[0]}")
    print(f"  all_docs loaded: {len(all_docs)}")

Extraction functions ready. Models: ['haiku', 'gpt4o-mini']
Samples available: 50
Test render: 3 pages, first page (1700, 2201)


In [ ]:
all_results = {}

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Running: {model_name} ({MODELS[model_name]['model_id']})")
    print(f"{'='*60}")

    results = []
    errors = []

    for i, s in enumerate(samples):
        predicted = None
        for attempt in range(5):
            try:
                predicted = extract_adbuy(s["pdf_path"], model_name)
                break
            except Exception as e:
                err_str = str(e)
                is_rate_limit = "429" in err_str or "rate_limit" in err_str.lower()
                if attempt < 4:
                    if is_rate_limit:
                        wait = min(10 * (2 ** attempt), 120)
                        print(f"  [{s['doc_id']}] Rate limited (attempt {attempt+1}). Waiting {wait}s...")
                    else:
                        wait = 2 ** (attempt + 1)
                        print(f"  [{s['doc_id']}] Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"  [{s['doc_id']}] FAILED after 5 attempts: {e}")
                    errors.append({"doc_id": s["doc_id"], "error": str(e)})

        if predicted is None:
            continue

        scores = score_document(s["ground_truth"], predicted)
        results.append({
            "doc_id": s["doc_id"],
            "filename": s["filename"],
            "predicted": predicted,
            "ground_truth": s["ground_truth"],
            "scores": scores,
        })

        mb = scores["misbinding_count"]
        status = "OK" if mb == 0 else f"MISBIND={mb}"
        if (i + 1) % 10 == 0 or i == 0 or mb > 0:
            print(f"  [{i+1:2d}/{len(samples)}] {s['doc_id']} — F1={scores['field_recall']:.3f}  CBA={scores['cba_strict']:.3f}  scored={scores['total']}  {status}")

        # Rate limit management
        time.sleep(1.5 if MODELS[model_name]["provider"] == "anthropic" else 5.0)

    # Aggregate
    if results:
        avg_f1 = sum(r["scores"]["field_recall"] for r in results) / len(results)
        avg_cba = sum(r["scores"]["cba_strict"] for r in results) / len(results)
        avg_cba_soft = sum(r["scores"]["cba_soft"] for r in results) / len(results)
        avg_delta = sum(r["scores"]["delta"] for r in results) / len(results)
        total_misbindings = sum(r["scores"]["misbinding_count"] for r in results)
    else:
        avg_f1 = avg_cba = avg_cba_soft = avg_delta = total_misbindings = 0

    all_results[model_name] = {
        "results": results,
        "errors": errors,
        "avg_f1": round(avg_f1, 4),
        "avg_cba": round(avg_cba, 4),
        "avg_cba_soft": round(avg_cba_soft, 4),
        "avg_delta": round(avg_delta, 4),
        "total_misbindings": total_misbindings,
    }

    print(f"\n--- {model_name} Summary ---")
    print(f"  Field Recall:     {avg_f1:.4f}")
    print(f"  CBA-strict:   {avg_cba:.4f}")
    print(f"  CBA-soft:     {avg_cba_soft:.4f}")
    print(f"  Delta:        {avg_delta:.4f}")
    print(f"  Misbindings:  {total_misbindings}")
    print(f"  Errors:       {len(errors)}")

print(f"\n{'='*60}")
print("ALL EXPERIMENTS COMPLETE")
print(f"{'='*60}")


Running: haiku (claude-haiku-4-5-20251001)
  [ 1/50] adbuy_0000 — F1=0.750  CBA=0.750  scored=8  OK
  [10/50] adbuy_0009 — F1=0.875  CBA=0.875  scored=8  OK
  [20/50] adbuy_0019 — F1=0.333  CBA=0.333  scored=9  OK
  [30/50] adbuy_0029 — F1=0.625  CBA=0.625  scored=8  OK
  [40/50] adbuy_0039 — F1=0.778  CBA=0.778  scored=9  OK
  [50/50] adbuy_0049 — F1=0.625  CBA=0.625  scored=8  OK

--- haiku Summary ---
  Field F1:     0.7623
  CBA-strict:   0.7623
  CBA-soft:     0.7623
  Delta:        0.0000
  Misbindings:  0
  Errors:       0

Running: gpt4o-mini (gpt-4o-mini)
  [ 1/50] adbuy_0000 — F1=0.500  CBA=0.500  scored=8  OK
  [adbuy_0008] Rate limited (attempt 1). Waiting 10s...
  [adbuy_0009] Rate limited (attempt 1). Waiting 10s...
  [10/50] adbuy_0009 — F1=0.750  CBA=0.750  scored=8  OK
  [adbuy_0011] Rate limited (attempt 1). Waiting 10s...
  [adbuy_0012] Rate limited (attempt 1). Waiting 10s...
  [adbuy_0013] Rate limited (attempt 1). Waiting 10s...
  [adbuy_0017] Rate limited (attem

In [ ]:
# Cross-model comparison
print("=" * 75)
print("CROSS-MODEL COMPARISON — VRDU Ad-buy Forms")
print("=" * 75)

print(f"\n{'Model':<15} {'Field Recall':>10} {'CBA-strict':>12} {'CBA-soft':>10} {'Delta':>8} {'Misbindings':>13} {'Errors':>8}")
print("-" * 80)
for name, res in all_results.items():
    print(f"{name:<15} {res['avg_f1']:>10.4f} {res['avg_cba']:>12.4f} {res['avg_cba_soft']:>10.4f} {res['avg_delta']:>8.4f} {res['total_misbindings']:>13} {len(res['errors']):>8}")

total_misbindings = sum(r["total_misbindings"] for r in all_results.values())
print(f"\nTotal misbindings across all models: {total_misbindings}")

# Per-concept accuracy
print(f"\n{'='*75}")
print("PER-CONCEPT ACCURACY")
print(f"{'='*75}")

concept_stats = defaultdict(lambda: {"cba": 0, "field": 0, "total": 0, "misbindings": 0})
for model_name, res in all_results.items():
    for r in res["results"]:
        for concept, detail in r["scores"]["per_field"].items():
            cs = concept_stats[concept]
            cs["total"] += 1
            if detail["cba_match"]:
                cs["cba"] += 1
            if detail["field_match"]:
                cs["field"] += 1
            if detail["misbinding"]:
                cs["misbindings"] += 1

concept_rows = []
for concept in EVAL_CONCEPTS:
    cs = concept_stats[concept]
    if cs["total"] == 0:
        continue
    concept_rows.append((concept, cs))

concept_rows.sort(key=lambda x: -x[1]["misbindings"])

print(f"\n{'Concept':<20} {'Family':<15} {'Field Acc':>10} {'CBA Acc':>10} {'Delta':>8} {'Misbindings':>13}")
print("-" * 80)
for concept, cs in concept_rows:
    f_acc = cs["field"] / cs["total"]
    c_acc = cs["cba"] / cs["total"]
    delta = f_acc - c_acc
    fam = CONCEPT_TO_FAMILY[concept]
    flag = " ***" if cs["misbindings"] > 3 else ""
    print(f"{concept:<20} {fam:<15} {f_acc:>10.3f} {c_acc:>10.3f} {delta:>8.3f} {cs['misbindings']:>13}{flag}")

CROSS-MODEL COMPARISON — VRDU Ad-buy Forms

Model             Field F1   CBA-strict   CBA-soft    Delta   Misbindings   Errors
--------------------------------------------------------------------------------
haiku               0.7623       0.7623     0.7623   0.0000             0        0
gpt4o-mini          0.6550       0.6525     0.6537   0.0025             1        0

Total misbindings across all models: 1

PER-CONCEPT ACCURACY

Concept              Family           Field Acc    CBA Acc    Delta   Misbindings
--------------------------------------------------------------------------------
advertiser           organizations        0.730      0.720    0.010             1
agency               organizations        0.618      0.618    0.000             0
contract_num         financial            0.760      0.760    0.000             0
flight_from          dates                0.733      0.733    0.000             0
flight_to            dates                0.767      0.767    0.000     

In [ ]:
# Misbinding confusion pairs
print("=" * 75)
print("MISBINDING CONFUSION PAIRS")
print("=" * 75)

confusion = Counter()
all_misbindings = []
for model_name, res in all_results.items():
    for r in res["results"]:
        for concept, detail in r["scores"]["per_field"].items():
            if detail["misbinding"] and detail["bound_to"]:
                confusion[(concept, detail["bound_to"])] += 1
                all_misbindings.append({
                    "model": model_name,
                    "doc_id": r["doc_id"],
                    "concept": concept,
                    "bound_to": detail["bound_to"],
                    "gt_value": detail["gt_value"],
                    "family_gt": CONCEPT_TO_FAMILY.get(concept, "?"),
                    "family_pred": CONCEPT_TO_FAMILY.get(detail["bound_to"], "?"),
                })

if confusion:
    print(f"\n{'Expected Concept':<20} {'Bound To':<20} {'Count':>6} {'Families':>30}")
    print("-" * 80)
    for (src, dst), count in confusion.most_common(20):
        fam_src = CONCEPT_TO_FAMILY.get(src, "?")
        fam_dst = CONCEPT_TO_FAMILY.get(dst, "?")
        same = "SAME-FAM" if fam_src == fam_dst else "CROSS-FAM"
        print(f"{src:<20} {dst:<20} {count:>6} {fam_src}->{fam_dst} ({same})")

    # Family-level summary
    print(f"\n{'='*75}")
    print("FAMILY-LEVEL CONFUSION SUMMARY")
    print(f"{'='*75}")
    fam_confusion = Counter()
    for mb in all_misbindings:
        fam_confusion[(mb["family_gt"], mb["family_pred"])] += 1

    print(f"\n{'GT Family':<20} {'Pred Family':<20} {'Count':>6} {'Type':>12}")
    print("-" * 60)
    for (fgt, fpred), count in fam_confusion.most_common():
        same = "within-fam" if fgt == fpred else "cross-fam"
        print(f"{fgt:<20} {fpred:<20} {count:>6} {same:>12}")
else:
    print("No misbindings detected.")

MISBINDING CONFUSION PAIRS

Expected Concept     Bound To              Count                       Families
--------------------------------------------------------------------------------
advertiser           product                   1 organizations->organizations (SAME-FAM)

FAMILY-LEVEL CONFUSION SUMMARY

GT Family            Pred Family           Count         Type
------------------------------------------------------------
organizations        organizations             1   within-fam


In [ ]:
export = {
    "experiment": "hindsight_vrdu_adbuy_cba",
    "domain": "political_advertising",
    "dataset": "VRDU ad-buy forms (google-research-datasets/vrdu)",
    "sample_size": len(samples),
    "total_concepts": len(EVAL_CONCEPTS),
    "ontology": ADBUY_ONTOLOGY,
    "models": {k: v for k, v in MODELS.items()},
    "scoring_method": "value-first CBA, GT-backed concepts only",
    "results": {
        model_name: {
            "field_recall": res["avg_f1"],
            "cba_strict": res["avg_cba"],
            "cba_soft": res["avg_cba_soft"],
            "delta": res["avg_delta"],
            "total_misbindings": res["total_misbindings"],
            "num_errors": len(res["errors"]),
        }
        for model_name, res in all_results.items()
    },
    "confusion_pairs": [
        {"expected": src, "bound_to": dst, "count": count}
        for (src, dst), count in confusion.most_common(20)
    ] if confusion else [],
    "total_misbindings": total_misbindings,
}

output_path = "/tmp/hindsight_vrdu_adbuy_cba_results.json"
with open(output_path, "w") as f:
    json.dump(export, f, indent=2)
print(f"Results exported to: {output_path}")

# Cross-domain comparison
print(f"\n{'='*75}")
print("CROSS-DOMAIN COMPARISON (all experiments)")
print(f"{'='*75}")
print(f"\n{'Domain':<25} {'Dataset':<20} {'Misbindings':>12} {'Haiku Delta':>13} {'GPT4o-mini Delta':>17}")
print("-" * 90)
print(f"{'Paystubs':<25} {'Synthetic':<20} {'~5':>12} {'~0':>13} {'~0':>17}")
print(f"{'Receipts (SROIE)':<25} {'SROIE':<20} {'3':>12} {'0.000':>13} {'+0.015':>17}")
print(f"{'Receipts (CORD)':<25} {'CORD v2':<20} {'48':>12} {'-0.10':>13} {'~0':>17}")
print(f"{'W-2 Tax Forms':<25} {'Synthetic W-2':<20} {'421':>12} {'+0.128':>13} {'+0.094':>17}")
print(f"{'Employment Law':<25} {'LEDGAR':<20} {'350':>12} {'+0.207':>13} {'+0.260':>17}")
print(f"{'Commercial Law':<25} {'CUAD v1':<20} {'434':>12} {'+0.244':>13} {'+0.335':>17}")

haiku_d = all_results.get("haiku", {}).get("avg_delta", 0)
gpt_d = all_results.get("gpt4o-mini", {}).get("avg_delta", 0)
print(f"{'Political Ad-buy':<25} {'VRDU':<20} {total_misbindings:>12} {f'+{haiku_d:.3f}':>13} {f'+{gpt_d:.3f}':>17}")

Results exported to: /tmp/hindsight_vrdu_adbuy_cba_results.json

CROSS-DOMAIN COMPARISON (all experiments)

Domain                    Dataset               Misbindings   Haiku Delta  GPT4o-mini Delta
------------------------------------------------------------------------------------------
Paystubs                  Synthetic                      ~5            ~0                ~0
Receipts (SROIE)          SROIE                           3         0.000            +0.015
Receipts (CORD)           CORD v2                        48         -0.10                ~0
W-2 Tax Forms             Synthetic W-2                 421        +0.128            +0.094
Employment Law            LEDGAR                        350        +0.207            +0.260
Commercial Law            CUAD v1                       434        +0.244            +0.335
Political Ad-buy          VRDU                            1        +0.000            +0.003
